# 03 — Baseline Model: Predicted Desirability from Signup Profiles

Implements the popularity baseline specified in `report.qmd` (sec-baseline): candidates are
ranked by their **desirability** (acceptance rate), ignoring who is choosing. This notebook
builds and evaluates that baseline under the same leave-one-wave-out protocol as `sec-eval`.

Runs top to bottom from a cold kernel, seed `SEED` (imported from `sd_features`).

**Why "predicted" desirability, not looked-up desirability.** The obvious version of a
popularity baseline — rank people by how often they've actually been accepted — cannot be
computed here. Every participant attends exactly one wave, so under leave-one-wave-out CV,
every held-out participant has **zero training history** (verified in `02_Speed_Dating_Feature_Engineering.ipynb`,
section 11: 0 of 551 held-out participants have any prior encounters in any of the 21 folds).
"Rank by real popularity" is undefined for everyone we need to rank.

The fix: train a model on the 20 training waves that predicts a participant's desirability
*from their signup profile alone* (age, interests, self-ratings, stated preferences — nothing
about any specific partner), then apply it to the held-out wave's participants. This keeps the
baseline non-personalized and global (the defining property of a popularity ranking) while
remaining computable under a total cold start.

**The target.** `report.qmd` specifies a **Wilson lower bound** on acceptance rate as the
desirability measure, so a participant accepted 4/4 times doesn't outrank one accepted 40/50
times just because they had fewer dates. `sd_features.add_behavioural_features()` already has a
shrinkage function for the same underlying idea, but it's additive/Bayesian shrinkage toward the
training-wide average — a different formula, not the one the report describes. That mismatch was
flagged in PR review; `sd_features.wilson_desirability()` is the version this notebook actually
uses, and it's now the single source of truth for both.

In [1]:
import sys
sys.path.insert(0, "..") if ".." not in sys.path else None

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from sd_features import (
    CATEGORY_LEVELS, INTERESTS, PREF, PROFILE_CAT, PROFILE_NUM, SEED, SELFRATE,
    build_person_table, load_raw, wilson_desirability as _wilson_desirability_features,
)

pd.set_option("display.width", 120)
K = 3                    # the group's decided evaluation cutoff (candidate lists: median 16 people)
N_RANDOM_REPEATS = 50    # repeats per fold for the chance-level estimate
print(f"K = {K}, SEED = {SEED}")

K = 3, SEED = 42


## 1. Confirm the cold start (why this baseline can't just look up real history)

Quick sanity check before building anything: for every wave held out, how many of that wave's
participants have *any* encounter in the training waves? This should be zero for all 21 folds —
that's the whole justification for predicting desirability instead of looking it up.

In [2]:
raw = load_raw()

cold_start_check = []
for w in sorted(raw["wave"].dropna().unique()):
    train_iids = set(raw.loc[raw["wave"] != w, "iid"].unique())
    test_iids = set(raw.loc[raw["wave"] == w, "iid"].unique())
    cold_start_check.append({"wave": w, "test_participants_seen_in_training": len(test_iids & train_iids)})

cold_start_df = pd.DataFrame(cold_start_check)
print(f"Total held-out participants with ANY training history, across all 21 folds: "
      f"{cold_start_df['test_participants_seen_in_training'].sum()}")
cold_start_df.T

Total held-out participants with ANY training history, across all 21 folds: 0


,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
wave,1,2,3,4,5,6,7,8,9,10,...,12,13,14,15,16,17,18,19,20,21
test_participants_seen_in_training,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 2. Feature set: signup profile only

This model predicts a person's own desirability from their own signup answers — nothing about
any specific partner. `PROFILE_NUMERIC` / `PROFILE_CATEGORICAL` below are the own-side-only
columns (no `_p` partner columns) from `sd_features`.

In [3]:
PROFILE_NUMERIC = PROFILE_NUM + INTERESTS + SELFRATE + PREF
PROFILE_CATEGORICAL = PROFILE_CAT
print("Numeric features:", PROFILE_NUMERIC)
print("Categorical features:", PROFILE_CATEGORICAL)

Numeric features: ['age', 'imprace', 'imprelig', 'exphappy', 'go_out', 'date', 'sports', 'tvsports', 'exercise', 'dining', 'museums', 'art', 'hiking', 'gaming', 'clubbing', 'reading', 'tv', 'theater', 'movies', 'concerts', 'music', 'shopping', 'yoga', 'selfrate_attr', 'selfrate_sinc', 'selfrate_intel', 'selfrate_fun', 'selfrate_amb', 'pref_attr', 'pref_sinc', 'pref_intel', 'pref_fun', 'pref_amb', 'pref_shar']
Categorical features: ['race', 'field_cd', 'career_c', 'goal']


## 3. The Wilson-shrunk desirability target

Thin wrapper around `sd_features.wilson_desirability()`, shaped for this notebook's per-fold
train/apply split: fit on training-wave rows only, read off one number per iid.

In [4]:
def wilson_desirability(raw: pd.DataFrame, iids: np.ndarray) -> pd.Series:
    """Wilson lower-bound acceptance rate RECEIVED (dec_o) for each iid, using only
    that person's own encounters."""
    train_df = raw.loc[raw["iid"].isin(iids)]
    out = _wilson_desirability_features(train_df, train_df)
    return out.drop_duplicates("iid").set_index("iid")["desirability_wilson"]


def make_profile_preprocessor() -> ColumnTransformer:
    numeric = Pipeline([("impute", SimpleImputer(strategy="median"))])
    categorical = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            categories=[CATEGORY_LEVELS[c] for c in PROFILE_CATEGORICAL],
            handle_unknown="ignore", sparse_output=False,
        )),
    ])
    return ColumnTransformer([
        ("num", numeric, PROFILE_NUMERIC),
        ("cat", categorical, PROFILE_CATEGORICAL),
    ])

## 4. Ranking metrics: Recall@K, NDCG@K

Both required by `report.qmd`'s eval protocol (at least two relevance metrics). Relevance is
`match == 1` (mutual yes), not a one-sided `dec`.

In [5]:
def recall_at_k(ranked: list, relevant: set, k: int) -> float:
    if not relevant:
        return np.nan
    return len(set(ranked[:k]) & relevant) / len(relevant)


def ndcg_at_k(ranked: list, relevant: set, k: int) -> float:
    if not relevant:
        return np.nan
    dcg = sum(1.0 / np.log2(i + 2) for i, p in enumerate(ranked[:k]) if p in relevant)
    ideal_hits = min(len(relevant), k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else np.nan

## 5. Randomized tie-breaking

Predicted desirability turns out to be heavily compressed toward the mean (see the signal
diagnostic in section 8), so ties in predicted score are common. Candidates are shuffled per
decider before sorting, so ties break randomly and reproducibly instead of by whatever order
rows happen to sit in the raw data.

In [6]:
def _rank_by_score(g: pd.DataFrame, rng: np.random.Generator) -> list:
    shuffled = g.sample(frac=1.0, random_state=rng.integers(0, 2**32 - 1))
    return shuffled.sort_values("score", ascending=False, kind="stable")["pid"].tolist()

## 6. The leave-one-wave-out loop

For each wave held out: fit a `RandomForestRegressor` (profile features → Wilson-shrunk
desirability) on the training waves, predict desirability for the held-out wave's participants,
then rank each participant's met candidates by the *candidate's own* predicted desirability —
this is what keeps it a non-personalized popularity baseline, the score never depends on who's
asking.

**Hyperparameters** (300 trees, `max_depth=6`, `min_samples_leaf=5`) are fixed by convention, not
tuned. Tuning a baseline against the evaluation metric would turn it into a hidden second
"improved method" rather than a fair, simple reference point — the whole reason a baseline exists
is to *not* be optimized.

Two diagnostics are computed alongside the ranking metrics:
- **Signal diagnostic**: correlation between predicted and *true* held-out-wave desirability, and
  the ratio of their spreads. Computed transductively within the held-out wave for this check
  only — never used to build the ranking itself, so it isn't leakage into the baseline, just a
  sanity check on it.
- **Chance-level Recall@K/NDCG@K**: the same candidates in fully random order, averaged over
  `N_RANDOM_REPEATS` shuffles, so "beats chance" is demonstrated rather than assumed.

In [7]:
def run_leave_one_wave_out(raw: pd.DataFrame, k: int = K, seed: int = SEED) -> pd.DataFrame:
    person = build_person_table(raw)
    waves = sorted(raw["wave"].dropna().unique())
    rows = []
    rng = np.random.default_rng(seed)

    for w in waves:
        train_iids = raw.loc[raw["wave"] != w, "iid"].unique()
        test_iids = raw.loc[raw["wave"] == w, "iid"].unique()

        y_train = wilson_desirability(raw, train_iids).dropna()
        X_train = person.loc[person.index.isin(y_train.index)]
        y_train = y_train.loc[X_train.index]

        prep = make_profile_preprocessor()
        X_train_t = prep.fit_transform(X_train)
        model = RandomForestRegressor(n_estimators=300, max_depth=6, min_samples_leaf=5, random_state=seed)
        model.fit(X_train_t, y_train.to_numpy())

        X_test = person.loc[person.index.isin(test_iids)]
        X_test_t = prep.transform(X_test)
        pred_desirability = pd.Series(model.predict(X_test_t), index=X_test.index)

        # signal diagnostic (not used for ranking -- transductive, held-out-wave only)
        y_test_true = wilson_desirability(raw, test_iids).dropna()
        common = X_test.index.intersection(y_test_true.index)
        if len(common) > 2:
            signal_corr = np.corrcoef(pred_desirability.loc[common], y_test_true.loc[common])[0, 1]
        else:
            signal_corr = np.nan
        pred_std = pred_desirability.loc[common].std() if len(common) else np.nan
        true_std = y_test_true.loc[common].std() if len(common) else np.nan

        test_pairs = raw.loc[(raw["wave"] == w) & raw["pid"].notna(), ["iid", "pid", "match"]].copy()
        test_pairs["pid"] = test_pairs["pid"].astype("int64")
        test_pairs["score"] = test_pairs["pid"].map(pred_desirability)

        recalls, ndcgs, topk_union = [], [], set()
        random_recalls, random_ndcgs = [], []
        for _, g in test_pairs.groupby("iid"):
            ranked = _rank_by_score(g, rng)
            relevant = set(g.loc[g["match"] == 1, "pid"])

            r, n = recall_at_k(ranked, relevant, k), ndcg_at_k(ranked, relevant, k)
            if not np.isnan(r):
                recalls.append(r)
            if not np.isnan(n):
                ndcgs.append(n)
            topk_union.update(ranked[:k])

            if relevant:
                pids = g["pid"].tolist()
                rr, rn = [], []
                for _ in range(N_RANDOM_REPEATS):
                    perm = rng.permutation(pids).tolist()
                    rr.append(recall_at_k(perm, relevant, k))
                    rn.append(ndcg_at_k(perm, relevant, k))
                random_recalls.append(np.mean(rr))
                random_ndcgs.append(np.mean(rn))

        coverage = len(topk_union) / test_pairs["pid"].nunique()
        rows.append({
            "wave": w,
            "n_test_participants": len(test_iids),
            f"recall@{k}": np.mean(recalls) if recalls else np.nan,
            f"ndcg@{k}": np.mean(ndcgs) if ndcgs else np.nan,
            "coverage": coverage,
            f"chance_recall@{k}": np.mean(random_recalls) if random_recalls else np.nan,
            f"chance_ndcg@{k}": np.mean(random_ndcgs) if random_ndcgs else np.nan,
            "signal_corr": signal_corr,
            "pred_std": pred_std,
            "true_std": true_std,
        })

    return pd.DataFrame(rows)

results = run_leave_one_wave_out(raw)
results.to_csv("baseline_results_by_wave.csv", index=False)
results

,wave,n_test_participants,recall@3,ndcg@3,coverage,chance_recall@3,chance_ndcg@3,signal_corr,pred_std,true_std
0,1,20,0.623347,0.692738,0.300000,0.305068,0.397949,0.444892,0.034720,0.167233
1,2,35,0.121333,0.162963,0.171429,0.187093,0.184548,0.338931,0.041953,0.201073
2,3,20,0.288889,0.243624,0.300000,0.270667,0.229674,0.297431,0.045286,0.136178
3,4,36,0.139597,0.252884,0.166667,0.165478,0.243700,0.182402,0.051683,0.195248
4,5,19,0.407460,0.538129,0.315789,0.323216,0.408452,0.233576,0.042195,0.181364
5,6,10,0.812500,0.617188,0.600000,0.587500,0.447686,0.577822,0.042799,0.158209
6,7,32,0.193367,0.236421,0.187500,0.187537,0.216572,0.599165,0.046861,0.185467
7,8,20,0.222549,0.165710,0.300000,0.298745,0.272088,-0.084345,0.037119,0.182368
8,9,40,0.251263,0.241938,0.150000,0.153901,0.203011,0.407491,0.042656,0.232696
9,10,18,0.500000,0.537909,0.333333,0.371095,0.343536,0.511896,0.053402,0.211539


## 7. Results: mean ± std across the 21 folds

In [8]:
metric_cols = [f"recall@{K}", f"ndcg@{K}", "coverage", f"chance_recall@{K}", f"chance_ndcg@{K}"]
summary = results[metric_cols].agg(["mean", "std"])
summary

,recall@3,ndcg@3,coverage,chance_recall@3,chance_ndcg@3
mean,0.350248,0.348146,0.275913,0.277920,0.278968
std,0.216405,0.179131,0.130772,0.129026,0.086969


## 8. Signal diagnostic — does predicted desirability track the truth?

This is the check that shows *why* the baseline lands where it does, rather than just reporting
a number. If predicted desirability barely correlates with real desirability, or the model's
predictions barely vary across people, that explains a modest Recall@K honestly instead of
leaving it as an unexamined result.

In [9]:
print(f"Mean corr(predicted, true Wilson desirability): "
      f"{results['signal_corr'].mean():.3f} (std {results['signal_corr'].std():.3f})")
print(f"Mean predicted-score std dev per fold: {results['pred_std'].mean():.4f}")
print(f"Mean true-desirability std dev per fold: {results['true_std'].mean():.4f}")
print(f"Ratio (predicted spread / true spread): "
      f"{results['pred_std'].mean() / results['true_std'].mean():.3f}")

Mean corr(predicted, true Wilson desirability): 0.270 (std 0.183)
Mean predicted-score std dev per fold: 0.0434
Mean true-desirability std dev per fold: 0.1742
Ratio (predicted spread / true spread): 0.249


**Reading this**: correlation ≈ 0.27, and predicted scores vary only about a quarter as much as
real desirability does. The model is playing it safe — predicting close to "roughly average" for
most people — rather than confidently differentiating them. That's consistent with the EDA's own
finding that admissible pre-date features correlate weakly with outcomes (|r| ≈ 0.067 at best,
`01_Speed_Dating_EDA.ipynb` section 13). It's not a bug in the model; it's an honest reflection of
how little signal a signup form actually carries about how popular someone will turn out to be.

## 9. Does it beat chance? (paired significance test across the 21 folds)

Coverage and Recall@3/NDCG@3 alone don't say whether this baseline is doing anything beyond
random ranking. Since both the model's metrics and the chance-level metrics are computed on the
exact same 21 folds, a **paired** test is the right comparison.

In [10]:
print("=== Does the model beat chance? (paired across the 21 folds) ===")
for metric, chance_col in [(f"recall@{K}", f"chance_recall@{K}"), (f"ndcg@{K}", f"chance_ndcg@{K}")]:
    diff = results[metric] - results[chance_col]
    t_stat, p_t = stats.ttest_rel(results[metric], results[chance_col])
    _, p_w = stats.wilcoxon(results[metric], results[chance_col])
    print(
        f"{metric}: model={results[metric].mean():.3f} vs chance={results[chance_col].mean():.3f}, "
        f"mean paired diff={diff.mean():.3f}, beats chance in {(diff > 0).sum()}/{len(diff)} folds, "
        f"paired t-test p={p_t:.4f}, wilcoxon p={p_w:.4f}"
    )

=== Does the model beat chance? (paired across the 21 folds) ===
recall@3: model=0.350 vs chance=0.278, mean paired diff=0.072, beats chance in 14/21 folds, paired t-test p=0.0122, wilcoxon p=0.0216
ndcg@3: model=0.348 vs chance=0.279, mean paired diff=0.069, beats chance in 16/21 folds, paired t-test p=0.0090, wilcoxon p=0.0080


**Reading this**: the model beats chance, and the gap is statistically real (p ≈ 0.01–0.02 on
both tests) — but it's a modest gap (~0.07 absolute), not a large one. Put together with section
8, the honest story is: this baseline captures a real, small, above-chance regularity in who
tends to be desirable from their signup form alone, and that modesty is exactly what a fair
baseline should look like — clearly beatable by a method that actually models reciprocal
compatibility, not a strawman.

## 10. Summary for `report.qmd`

These numbers (and `baseline_results_by_wave.csv`, exported above) are what fill the
`<!-- PLACEHOLDER: baseline results -->` marker in `sec-baseline`, reported under the same
21-fold protocol as `sec-eval` per the report's own requirement.

In [11]:
print("| Metric | Mean | Std dev |")
print("|---|---|---|")
print(f"| Recall@{K} | {results[f'recall@{K}'].mean():.3f} | {results[f'recall@{K}'].std():.3f} |")
print(f"| NDCG@{K} | {results[f'ndcg@{K}'].mean():.3f} | {results[f'ndcg@{K}'].std():.3f} |")
print(f"| Coverage | {results['coverage'].mean():.3f} | {results['coverage'].std():.3f} |")

| Metric | Mean | Std dev |
|---|---|---|
| Recall@3 | 0.350 | 0.216 |
| NDCG@3 | 0.348 | 0.179 |
| Coverage | 0.276 | 0.131 |
